In [2]:
# General libraries
import os, sys # system libraries
import ipdb # for debugging
from tqdm import tqdm # for the progress bar
from datetime import datetime # for the intermediate checkpoints on training
import platform, shutil # detect plataform type
import requests, zipfile, io # generic libraries

# Pytorch (standard tool for deep learning)
import torch
import torch.nn as nn
from torch.nn import functional as F # set of methods

# Tokenizer (convert human language to tokens)
import sentencepiece as spm

# Improve performance for Ampere architecture (precision type improvement for certain types of GPUs)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Empty GCPU Cache Memory
torch.cuda.empty_cache()

In [3]:
# Download the necessary files and create the necessary folders for the data
#    - wiki.txt dataset: a tiny segment of the english wikipedia
#    - wiki_tokenizer.model: trained tokenizer file
#    - wiki_tokenizer.vocab: trained tokenizer file
#    - encoded_data.pt: dataset tokenized with the tokenizer

In [4]:
files_url = "https://ideami.com/llm_train"
print("Downloading")
response = requests.get(files_url) # download the zip folder from the website
zipfile.ZipFile(io.BytesIO(response.content)).extractall(".") # unzip the folder

Downloading


In [26]:
# ARCHITECTURE PARAMETERS

batch_size = 8 # limit of batch for gpu with 4GB of memory (19M parameters)
context = 512 # window to learn about the relationship between tokens (the higher the best but its computational expensive)
embed_size = 384 # abstarct multidimension representation of the token (a vector of 384 numbers)

n_layers = 7 # Each of the seven layers contains:
             #    - Communication: attention mechanism that learns how the different tokens relate to each other (HEADS)
             #    - Computation: a layer that provides complex processing for the network (FEED-FORWARD)

n_heads = 7 # turns to a multi-head attention mechanism: an input arrives to the attention mechanism block adn gets divided into a number of attention
            # heads which will peocess part of that input. After all the heads processe theire outputs is combined.

# FOR each layer:
    # embedings -> divided into the number of heads -> each head follows the attention mechanism -> results combined -> feed-forward layer
# END

BIAS = True # aditional parameter to allow the activation function to shift

In [6]:
# HYPER PARAMETERS

lr = 3e-4 # learning rate: how fast the tweeking is done of the parameters (too fast - skip the best solutions, too slow - we might not get the good solution)
dropout = 0.05 # regularization to prevent overfit. "Disconect random neurons during the trainign phase".
weight_decay = 0.01 # prevent the size of the network weight from becoming too large
grad_clip = 1.0 # prevents exploding gradient by capping them to a maximum value for a stable and efficient learning

In [7]:
# TRAINING PARAMETERS

train_iters = 100000
eval_interval = 50 # every 50 iteration we take the evaluation data to test it to check for overfit
eval_iters = 10 # the evaluation of the loss with the evaluation data we use 10 iterations and average the perfomance
compile = False # pytorch command to lower the memory needed and a fast train
checkpoint_dir = "models/" # to store the intermediate models
checkpoint_fn = "latest.pt" # saving a checkpoint
checkpoint_load_fn = "latest.pt" # loaded a checkpoint and to continue to be trained
dtype = torch.bfloat16
load_pretrained = True

In [41]:
# GENERAL PARAMETERS

# MODE
inference = True

# DEVICE
device = "cuda" if torch.cuda.is_available() else "cpu"
print("You are using the device: ", device)

You are using the device:  cuda


In [9]:
# LOGGING
# Prepare to show graphs and plots during the processes.
# Use "Weights and Biases" free website to monitor the training processes (https://wandb.ai/home)

wandb_log = True
wandb_project = "my_llm"
wandb_run_name = "my_llm-" + datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

if wandb_log:
    import wandb
    wandb.init(project=wandb_project, name=wandb_run_name)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\andre\_netrc.
wandb: Currently logged in as: andre99amaral to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [10]:
# LOAD the INPUT DATA
with open("wiki.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [11]:
# LOAD the TOKENIZER
sp = spm.SentencePieceProcessor(model_file="wiki_tokenizer.model")
vocab_size = sp.get_piece_size()

In [12]:
# ENCODER - text to tokens
encode = lambda s: sp.Encode(s)

# DECODER - tokens to text
decode = lambda l: sp.Decode(l)

In [13]:
# CREATE the ENCODINGS
if os.path.exists(f"encoded_data.pt"):
    print("loading encoding")
    data = torch.load("encoded_data.pt")
else:
    print("creating the encoding")
    data = torch.tensor(encode(text), dtype=torch.long)
    torch.save(data, "encoded_data.pt")

loading encoding


In [14]:
# SPLIT the DATA
data_size = len(data)
spl = int(0.9*data_size)

train_data = data[:spl]
val_data = data[spl:]

In [15]:
def get_batch(split):
    
    data = train_data if split == "train" else val_data
    inds = torch.randint(len(data)-context, (batch_size,))
    
    # input batches
    x = torch.stack([data[i: i+context] for i in inds]) # extract 8 sequences and stack them into a batch, tensor(batch_size, context) tensor(8,512)
    
    # output batches
    y = torch.stack([data[i+1: i+context+1] for i in inds]) # +1 beacuse we want the output one position shifted related to the input

    x,y = x.to(device), y.to(device)

    return x,y

############################################### CREATION OF LLM MODEL #################################################

In [16]:
class GPT(nn.Module):
    
    # Declaration of the layers
    def __init__(self):
        super().__init__()

        ########### Initialize the layers########################
        
        #create embedding layer to transform tokens in a vector to capture the meaning of tokens (4096x384)
        self.embeddings = nn.Embedding(vocab_size, embed_size)

        #create embedding position layer to capture the position within the sequence of the token (512x384)
        self.positions = nn.Embedding(context, embed_size)

        #create the layers for the transformer (multihead attention (n_heads) + feedforward + normalization)
        self.blocks = nn.Sequential(*[Block(n_heads) for _ in range(n_layers)])

        #create layer for normalization (keep the numbers on a comfortable range for the llm to process them)
        self.ln = nn.LayerNorm(embed_size)

        #create layer for the final computation (384x40966) - probability of each of the 4096 tokens to be the next one
        self.final_linear = nn.Linear(embed_size, vocab_size, bias=BIAS)

        #apply to all layers the initialization
        self.apply(self._init_weights)
        

    # Parameters initialization
    def _init_weights(self, module):

        # Normal destribution for linear layers initialization, zeros for biases initialization
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            

    # Define the feed-forward function
    # BS = Batch Size, SL = Sequence or Context Length
    def forward(self, input, targets=None):
        loss = None
        BS, SL = input.shape # (BS x SL) = (8 x 512)

        # Propagation through the layers
        emb = self.embeddings(input) # (BS x SL x embed_size) = (8 x 512 x 384)
        pos = self.positions(torch.arange(SL, device = device)) # Embedding to each 512 positions (SL x embed_size) = (512 x 384)
        
        x = emb + pos #merge the embeddings with thch of the blocks (BS x SL x embed_size) = (8 x 512 x 384)
        x = self.blocks(x) #(BS x SL x embed_size)= (8 x 512 x 384)
        x = self.ln(x) #normalize the data (BS x SL x embede positions (BS x SL x embed_size) = (8 x 512 x 384)
        logits = self.final_linear(x) # (BS x SL x vocab_size) = (8 x 512 x 4096) logits represent the probabilities in nn language of next token


        # Loss computation
        # if we are in training mode
        if targets is not None:
            BS, SL, VS = logits.shape

            # Reshape the logits and targets to match pytorch needs
            logits = logits.view(BS*SL, VS) # p
            targets = targets.view(BS*SL) # q

            # Compute the loss using the cross entropy
            loss =  F.cross_entropy(logits, targets)

            #### Manual calculation of the cross entropy ####
            # 1) Turn the nn probabilites (logits) in real probabilities then compute the loss
            counts = logits.exp() #all numbers positive and exagerate the difference between the numbers
            prob = counts/counts.sum(-1, keepdim=True) # equivalent to a softmax function, with this we went from logits to probabilities
            # 2) Compute the cross entropy
            # Information: oposit of probability. High probability brings low information. Uncertainty brings more information
            #              i = -log(p) probability of 1 information is 0, probability of 0 information is very high
            # Entropy: average information of a random variable. H(x) = SUM(p(x) Log p(x))
            # Cross Entropy: H(p,q) = - SUM(q(x) * log p(x)) - the cross distribution between q (targets) p (predictions)
            #                The sum disaper because in the q (target) all values are 0 except one. H(p,q) = - (q(x) * log p(x))
            #                Then we know what is the right/correct token in the target H(p,q) = - (1 * log p(x)) = - log (p)
            # go to the targets and get the index of the correct target then we go to the predictions vector to that index and we extract
            # the probability and apply the -log to that probability and we get the cross entropy
            loss2 = - prob[torch.arange(BS*SL), targets].log().mean()

        return logits, loss         


    # Generate a new sample (after the LLM is trained)
    def generate(self, input, max=500):
        for _ in range(max):
            input = input[:,-context:] # taking always the last 512 tokens of our input. (1, input length until max of context)
            logits, _ = self(input) #produce the answer (1, input length, 4096)
            logits = logits[:,-1,:] #get the next word after the last element of the input sequence (1,4096)
            probs = F.softmax(logits, dim=-1) #convert logits into probabilities (1,4096)
            next = torch.multinomial(probs, num_samples=1) # sample the next token value (what gives variability in the llms answers)
            input = torch.cat((input,next), dim=1) # concatenate the next token to the input
        return input

In [17]:
# Create each block of the transformer (Attention + Computation)
class Block(nn.Module):
    
    def __init__(self, n_heads):
        super().__init__()
        head_size = embed_size // n_heads # Split the embeddings among each heads of the multihead attention mechanism
        self.ma = Multihead(n_heads, head_size)
        self.feed_forward = ForwardLayer(embed_size)
        self.ln1 = nn.LayerNorm(embed_size)
        self.ln2 = nn.LayerNorm(embed_size)
    
    def forward(self, x):
        # Deeper networks can be unstable and problems of vanishing gradients might appear
        # Having this residual connections the gradients can travell fast through the network.
        # Also help to learn identity mappings
        x = x + self.ma(self.ln1(x)) # attention
        x = x + self.feed_forward(self.ln2(x)) # computation
        return x     

In [18]:
class ForwardLayer(nn.Module):
    
    def __init__(self, embed_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(embed_size, 6*embed_size, bias = BIAS),
            nn.GELU(),
            nn.Linear(6*embed_size, embed_size, bias=BIAS),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)

In [19]:
class Multihead(nn.Module):
    
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)]) # Create a list of heads
        self.combine = nn.Linear(head_size*n_heads, embed_size, bias=BIAS)
        self.dropout= nn.Dropout(dropout)

    def forward(self, x):
        x = torch.cat([head(x) for head in self.heads], dim =-1) # Concatenate each head output
        # Each head outputs (BS, SL, head_size)
        x = self.combine(x) #(BS, SL, Embed_size)
        x = self.dropout(x)
        return x

In [20]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        # Projections of the embeddings:
        # 1) Determine the importance of different words in a sequence
        # 2) Three layers of projections give more insight in the possible combinations
        self.queries = nn.Linear(embed_size, head_size, bias=BIAS) # "Asking questions" - how aligned are the embedings of a token comparing to the other tokens
        self.keys = nn.Linear(embed_size, head_size, bias=BIAS) # provide the tokens to be confirmed through the querie
        self.values = nn.Linear(embed_size, head_size, bias=BIAS) # contain the content to be updated

        # Triangular Matrix - because what we need is the previous tokens not the future ones (mask out any knowledge about future tokens)
        self.register_buffer('tril', torch.tril(torch.ones(context, context)))
        self.dropout= nn.Dropout(dropout)

    def forward(self, x):
        BS, SL, VS = x.shape
        q = self.queries(x) # BS, SL, 54
        k = self.keys(x) # BS, SL, 54
        v = self.values(x) # BS, SL, 54

        # Attention weights mechanism
        attn_w = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # BS, SL, SL - multiplication of queries with keys and then a convetional normalization
        # row 1 degree of compatibility of a first token with all tokens
        # row 2 degree of compatibility of a second token with all tokens
        # row 3 degree of compatibility of a third token with all tokens
        attn_w = attn_w.masked_fill(self.tril[:SL,:SL] == 0, float('-inf')) # Mask the future degrees of compatibility
        attn_w = F.softmax(attn_w, dim=-1) # BS, SL, SL - turn into probabilities

        x = attn_w @ v # BS, SL, 54

        return x

############################################### SETUPS FOR TRAINING THE LLM MODEL #################################################

In [21]:
# 1) Instantiate the model and send it to the GPU
model = GPT()
model = model.to(dtype).to(device)

# 2) Compile the model using torch library to improve efficiency
if compile:
    print("Torch: compiling model")
    model = torch.compile(model)

# 3) Print number of parameters (GPT 3 - 175 Billions parameters)
print(sum(p.numel() for p in model.parameters()) / 1e6, "Million parameters")

# 4) Setup the optimizer
p_dict= {p_name: p for p_name, p in model.named_parameters() if p.requires_grad}

weight_decay_p= [p for n,p in p_dict.items() if p.dim() >= 2]
no_weight_decay_p= [p for n,p in p_dict.items() if p.dim() < 2]

optimizer_groups=[
    {'params':weight_decay_p,'weight_decay':weight_decay},
    {'params':no_weight_decay_p,'weight_decay':0.0},
]

optimizer = torch.optim.AdamW(optimizer_groups, lr=lr, betas=(0.9,0.99))

# 5) Setup the Scheduler for changing the learning rate
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, train_iters, eta_min=lr/10)

# 6) Set iterations begining
start_iteration = 0
best_val_loss = float('inf')

19.837954 Million parameters


In [37]:
@torch.no_grad()
def generate_sample(input_text):
    t1 = torch.tensor(encode(input_text), dtype=torch.long, device=device)
    t1 = t1[None,:]
    new_gen = model.generate(t1, max=64)[0].tolist() # just the first dimension (for a list needed for the decoder)
    result = decode(new_gen)
    print(f"{result}")

# Loss Average of training and evaluation
@torch.no_grad()
def calculate_loss():
    out={}
    model.eval()
    for split in ['train', 'eval']:
        l = torch.zeros(eval_iters)
        for i in range(eval_iters):
            x,y=get_batch(split)
            _, loss = model(x,y)
            l[i] = loss
        out[split]=l.mean().item()
    model.train()
    return out

# Loading Checkpoints
def load_checkpoint(path):
    print("LLM - loading model")
    checkpoint = torch.load(path)
    state_dict = checkpoint['model_state_dict']

    model.load_state_dict(state_dict)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    iteration = checkpoint['iteration']
    loss = checkpoint ['loss']
    print(f"Loaded iter {iteration} with loss {loss}")
    return iteration, loss

if os.path.exists(f"{checkpoint_dir}/{checkpoint_load_fn}") and load_pretrained:
    start_iteration, loss = load_checkpoint(checkpoint_dir + checkpoint_load_fn)
    best_val_loss = loss

LLM - loading model
Loaded iter 650 with loss 4.818749904632568


############################################### TRAINING LOOP #################################################

In [38]:
try:
    for i in tqdm(range(start_iteration, train_iters)):
        xb, yb = get_batch("train")
        logits, loss = model(xb,yb)
    
        # Evaluation of the loss
        if (i % eval_interval == 0 or i == train_iters -1):
            l=calculate_loss()
            print(f"\n{i}: train loss: {l['train']} | validation loss: {l['eval']}")
            generate_sample("Once upon a time")
    
            # If we have a better loss we save the checkpoint
            if l['eval'] < best_val_loss:
                best_val_loss = l['eval']
                print("[CHECKPOINT]: Saving with loss: ", best_val_loss)
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': best_val_loss,
                    'iteration': i,
                }, checkpoint_dir + checkpoint_fn)
    
            if wandb_log:
                wandb.log({
                    "loss/train": l['train'],
                    "loss/val": l['eval'],
                    "lr": scheduler.get_last_lr()[0],
                },
                step = i)
    
        optimizer.zero_grad(set_to_none = True) # Compute new gradients
        loss.backward() # propagate the loss
        nn.utils.clip_grad_norm_(model.parameters(), max_norm = grad_clip)
        optimizer.step()
        scheduler.step()
    
    if wandb_log:
        wandb.finish()

except KeyboardInterrupt:
    print("Training Interrupted. Cleaning up ... ... ...")

finally:
    # Release GPU memmory
    print("Releasing the GPU")
    torch.cuda.empty_cache()
    
torch.cuda.empty_cache()

  0%|          | 0/99350 [00:00<?, ?it/s]


650: train loss: 4.753125190734863 | validation loss: 4.828125
Once upon a time where nineceptland to recorda fought an Disney in the aff here (XSial Burneroosio pass the wiscicksgarily singer in 146). He star.
Pey is white (Mannaques that is included Dolles, km


  0%|          | 50/99350 [02:41<77:41:24,  2.82s/it]


700: train loss: 4.737500190734863 | validation loss: 4.896874904632568
Once upon a time made a universondhenleluce Credo Gte in the father far after the Hotareic of 2ther under of education in New 2 and Romanian eidberbtation.In the C. state preeat call the television conves of . The French City. P


  0%|          | 100/99350 [04:12<14:40:30,  1.88it/s]

...


23200: train loss: 2.640625 | validation loss: 2.817187547683716
Once upon a time Haiti away.

Because Beure

Because Locknom, popularly named Cost", H24-192 mile ten0 was a non-constroxt work of many users such as Locle, or imitating powers


 23%|██▎       | 22600/99350 [4:11:53<11:59:34,  1.78it/s]


23250: train loss: 2.7750000953674316 | validation loss: 2.6953125
Once upon a time back from going pastors she had gifts there.

Donald Reagan

Donald Reagan Acadellago (October 2, 1839 – May 7, 1989) was an American politician. Manuitarist Re


 23%|██▎       | 22639/99350 [4:12:20<14:15:01,  1.50it/s]


Training Interrupted. Cleaning up ... ... ...
Releasing the GPU


############################################### INFERENCE LOOP #################################################

In [43]:
# Start the inference loop
if inference == True:
    model.eval()
    while True:
        question = input("Enter text (q to quit): ")
        if question == "":
            continue
        if question == "q":
            break
        generate_sample(question)
    sys.exit()

Enter text (q to quit):  one day


one day before North Fox's 2013 hits.


John MacThe Little Lood

Fean Rabis "Digecond" Chair (1934 – March 24, 1994). He had a powerful


Enter text (q to quit):  q


SystemExit: 0

In [39]:
torch.cuda.empty_cache()
!nvidia-smi

Thu Aug 27 14:40:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.52                 Driver Version: 596.52         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   75C    P5             11W /   35W |     503MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----